In [0]:
%sql
create or replace temporary view g1 as 
select 
    order_id,
    city,
    zip_code
from olist_dataset.silver.customers as c
left join olist_dataset.silver.delivered_orders as o
on c.customer_id = o.customer_id


In [0]:
%sql
create or replace temporary view g2 as 
select 
    g1.order_id,
    g1.city,
    g1.zip_code,
    i.price,
    i.shipping_cost
from g1 
left join olist_dataset.silver.geolocation as g
on g1.city = g.city
left join olist_dataset.silver.items as i
on g1.order_id = i.order_id

In [0]:
%sql
create or replace temporary view success_g as 
select 
    city,
    zip_code,
    sum(price) as revenue,
    avg(price) as avg_order_price,
    round(avg(shipping_cost),2) as avg_shipping_cost,
    count(order_id) as total_orders
from g2
group by city, zip_code


In [0]:
%sql
create or replace temporary view final_success as 
select  
    cast(g.zip_code_prefix as string) as zip_code,
    g.latitude,
    g.longitude,
    g.city,
    g.state,
    s.revenue,
    s.avg_order_price,
    s.avg_shipping_cost,
    s.total_orders
from olist_dataset.silver.geolocation as g
left join success_g as s 
on g.city = s.city and s.zip_code = g.zip_code_prefix

In [0]:
%sql
create or replace temporary view ga as 
select 
    order_id,
    zip_code,
    city
from olist_dataset.silver.undelivered_orders as o
left join olist_dataset.silver.customers as c 
on c.customer_id = o.customer_id

In [0]:
%sql
create or replace temporary view gb as 
select 
    ga.order_id,
    ga.city,
    ga.zip_code,
    i.price,
    i.shipping_cost
from ga
left join olist_dataset.silver.geolocation as g
on ga.city = g.city
left join olist_dataset.silver.items as i
on ga.order_id = i.order_id

In [0]:
%sql
create or replace temporary view final_failure as 
select 
    city,
    zip_code,
    count(order_id) as unsuccessful_orders
from gb
group by city, zip_code

In [0]:
%sql
create or replace temporary view geoloc as

select 
    cast(s.zip_code as string) as zip_code,
    s.latitude,
    s.longitude,
    s.city,
    s.state,
    s.revenue,
    round(s.avg_order_price,2) as avg_order_price,
    round(s.avg_shipping_cost,2) as avg_shipping_cost,
    coalesce(s.total_orders,0) as total_orders,
    coalesce(f.unsuccessful_orders, 0 ) as unsuccessful_orders,
    coalesce(round(coalesce(f.unsuccessful_orders, 0 ) / s.total_orders,2),0) as failure_percentage
from final_success as s 
left join final_failure as f 
on s.zip_code = f.zip_code and s.city = f.city

In [0]:
df = spark.read.table('geoloc')

df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable('olist_dataset.gold.geolocation')